# Laboratorio: Laplace en 2D, cinco puntos contra Monte Carlo, e inpainting

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jfbonder/imc/blob/main/notebooks/lab-laplace-datos.ipynb)

En el Capítulo 19 vimos que el problema de Dirichlet para la ecuación de Laplace, $\Delta u = 0$ en $\Omega$ con $u = g$ en $\partial\Omega$, se puede mirar de tres maneras que dicen lo mismo: como **ecuación de promedios** (el valor en cada punto es el promedio de sus vecinos), como **juego** (el valor esperado del pago $g$ de un paseo al azar que empieza en el punto y termina al tocar el borde) y como **problema variacional** (la función con menos energía $\int|\nabla u|^2$ que vale $g$ en el borde). Cada mirada da un algoritmo distinto, y cada algoritmo tiene su costo, sus errores y su lugar. Este laboratorio los pone a competir y, al final, los usa para algo que se ve: **rellenar un agujero en una fotografía**.

**Qué vamos a hacer.** Ensamblar el esquema de cinco puntos como un sistema lineal ralo, resolverlo y verificarlo contra la solución exacta por series de Fourier (Tarea 1); medir cuánto cuesta en tiempo y memoria, denso contra ralo (Tarea 2); resolver "jugando", con Monte Carlo, y medir cómo baja el error con el número de partidas (Tarea 3); comparar los costos de los dos métodos y ver qué pasa en dimensión alta (Tarea 4); resolver el sistema con métodos iterativos que son literalmente "promediar una y otra vez" y contar cuántas iteraciones hacen falta (Tarea 5); agregar una fuente, $-\Delta u = f$, y ver cómo cambia el juego (Tarea 6); y por último el **inpainting**: borrar una región de una imagen y rellenarla resolviendo Laplace con los píxeles del borde de la región como dato, en un fondo liso y en una textura (Tarea 7). La Tarea 8, opcional, usa una fuente $f$ construida a partir de otra parte de la imagen (clonado de textura, "Poisson image editing").

**Lo que las notas no explican y este notebook sí.** Cómo se ensambla el esquema de cinco puntos como matriz (con `scipy.sparse`) y qué cuesta resolverlo; los métodos iterativos (Jacobi, Gauss–Seidel, gradiente conjugado) y por qué los primeros son lentos; el error estadístico de Monte Carlo y su costo, comparado con el error de discretización; el juego con fuente (la versión discreta de la fórmula de Feynman–Kac); y la formulación del inpainting como un sistema lineal ralo sobre los píxeles de una región de forma arbitraria. El texto de las secciones 1 a 6 es la única presentación que van a tener de estos temas: léanlo, no solo las consignas.

**Herramientas disponibles.** `scipy.sparse` y `scipy.sparse.linalg`, `numpy`, `imc.numerico.laplaciano_2d` y `imc.numerico.poisson_cinco_puntos` (**para verificar** lo que ensamblen ustedes, no para reemplazarlo), `imc.datos.obtener` (la imagen `astronauta.png`), `imc.estilo`. El notebook `10-laplace` de las notas ya hizo, en $n=20$, la comparación cualitativa entre Monte Carlo y cinco puntos, el principio del máximo y el valor medio; acá lo hacemos en serio: con escalas, costos y verificaciones cuantitativas.

**Cómo se evalúa.** Como siempre: la sección final de **interpretación escrita**. Cada tarea dice qué se espera y trae una celda de verificación. Tiempo estimado: una sesión de 4 h para las Tareas 1 a 7; la Tarea 8 es opcional.

In [ ]:
# Configuración (funciona en Colab y en una copia local del repositorio)
try:
    import imc
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/jfbonder/imc.git"], check=True)
    import imc

import numpy as np
import matplotlib.pyplot as plt
from imc import estilo, datos
from imc.estilo import COLORES
import time
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from imc import numerico
from imc.estilo import CICLO
estilo.activar()

In [ ]:
# Convenciones del laboratorio (esta celda viene dada)
# Cuadrado unitario, n intervalos por lado, paso h = 1/n; u[i, j] es el valor en (x_j, y_i) = (j h, i h):
# la fila i = n es el lado superior (y = 1) y la columna j = n el lado derecho (x = 1).
# Un "dato de borde" G es un array (n+1, n+1) del que solo se usan los valores del borde (sin las esquinas).

def mapa_color(ax, U, titulo="", vmin=None, vmax=None, cmap="viridis"):
    """Dibuja el array U (n+1, n+1) como mapa de color sobre el cuadrado unitario."""
    h = 1 / (U.shape[0] - 1)
    im = ax.imshow(U, origin="lower", extent=[-h / 2, 1 + h / 2, -h / 2, 1 + h / 2], vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_title(titulo)
    return im

## 1. El esquema de cinco puntos: promedios, matriz y solución exacta

### La ecuación de promedios

Sea $h = 1/n$ y los nodos $(x_j, y_i) = (jh, ih)$, $0\le i,j\le n$. El desarrollo de Taylor da, para $u$ suave,

$$u(x+h,y) + u(x-h,y) - 2u(x,y) = h^2 u_{xx} + \tfrac{h^4}{12}u_{xxxx} + O(h^6),$$

y lo mismo en $y$. Sumando, el **Laplaciano discreto de cinco puntos** es

$$\Delta_h u\,(x,y) = \frac{u(x+h,y) + u(x-h,y) + u(x,y+h) + u(x,y-h) - 4u(x,y)}{h^2} = \Delta u + \frac{h^2}{12}\,(u_{xxxx} + u_{yyyy}) + O(h^4).$$

El error de consistencia es $O(h^2)$: **se espera orden 2**. La ecuación discreta $\Delta_h u = 0$ en un nodo interior dice exactamente

$$u_{i,j} = \frac{1}{4}\bigl(u_{i+1,j} + u_{i-1,j} + u_{i,j+1} + u_{i,j-1}\bigr),$$

es decir, **cada valor es el promedio de sus cuatro vecinos**. Esa es la ecuación de promedios del juego aleatorio, sin ningún límite: el esquema de cinco puntos *es* la ecuación de promedios. Con una fuente, $-\Delta_h u = f$ se lee $u_{i,j} = \frac14\sum(\text{vecinos}) + \frac{h^2}{4}f_{i,j}$ (Tarea 6).

### Como sistema lineal

Hay $m = n-1$ nodos interiores por lado, $N = m^2$ incógnitas en total. Los vecinos que caen en el borde no son incógnitas: pasan al lado derecho. Ordenamos las incógnitas **por filas**, $k = (i-1)\,m + (j-1)$ para $1\le i,j\le m$ (es el orden de `u[1:-1, 1:-1].ravel()`). Con la matriz tridiagonal $T = \frac{1}{h^2}\,\text{tridiag}(-1, 2, -1)\in\mathbb R^{m\times m}$ (la del problema unidimensional), el operador $-\Delta_h$ es

$$A = I_m\otimes T + T\otimes I_m \in\mathbb R^{N\times N},\qquad A\,\mathbf u = \mathbf b,$$

donde $\otimes$ es el producto de Kronecker (`scipy.sparse.kron`): $I_m\otimes T$ es la segunda diferencia en $j$ (una copia de $T$ por cada fila $i$) y $T\otimes I_m$ acopla las filas. $A$ tiene a lo sumo 5 entradas no nulas por fila (diagonal $4/h^2$ y cuatro vecinos $-1/h^2$), es simétrica y definida positiva. El lado derecho es $b_{i,j} = f_{i,j} + \frac{1}{h^2}\sum(\text{valores de } g \text{ de los vecinos que están en el borde})$: por ejemplo, el nodo $(1, j)$ tiene a $(0, j)$ como vecino, y suma $g_{0,j}/h^2$. Las esquinas del cuadrado no son vecinas de ningún nodo interior: el esquema no usa el valor de $g$ en ellas.

### Una solución exacta para comparar: la serie de Fourier

Para $g = 1$ en el lado superior ($y=1$) y $0$ en los otros tres lados, el método de separación de variables da

$$u(x,y) = \sum_{k\ \text{impar}} \frac{4}{k\pi}\,\sin(k\pi x)\,\frac{\sinh(k\pi y)}{\sinh(k\pi)} .$$

Dos detalles numéricos. (i) $\sinh(k\pi y)/\sinh(k\pi)$ se desborda para $k$ grande: escribilo como $e^{k\pi(y-1)}\,\dfrac{1 - e^{-2k\pi y}}{1 - e^{-2k\pi}}$, que no se desborda nunca. (ii) La serie converge rápido para $y<1$ (el factor decae como $e^{-k\pi(1-y)}$), pero el dato es **discontinuo en las dos esquinas de arriba** ($g$ salta de $1$ a $0$): cerca de $y = 1$ hacen falta muchos términos (`K` grande), y la solución exacta no es suave en las esquinas.

### Cómo medir el error, y qué esperar

Con $n = 20, 40, 80$ los nodos de la grilla gruesa están en las grillas finas, así que se puede comparar en **los mismos puntos**. Si $e_n$ es el error máximo en esos puntos, el **orden observado** es $p = \log_2(e_n / e_{2n})$: si el error es $\approx Ch^p$, cada vez que duplicás $n$ el error se divide por $2^p$. Lo que hay que mirar es **dónde** se mide el error, porque la singularidad de las esquinas contamina: medí (i) el error máximo en los nodos con $\tfrac14\le x,y\le\tfrac34$ (lejos del borde), (ii) el error cuadrático medio en todos los nodos interiores y (iii) el error máximo en todos los nodos interiores. **Qué se espera:** (i) orden $2$; (ii) un orden menor (¿cuánto?); (iii) que **no baje**: el error máximo sobre toda la grilla queda clavado en un valor de unos $7\times10^{-3}$, en los nodos vecinos a las esquinas donde el dato salta. Esa diferencia entre "el error en un punto fijo del interior" y "el error en todos lados" es un hecho general de las ecuaciones elípticas con datos poco regulares.

### El principio del máximo como diagnóstico

Si $\Delta_h u = 0$ en el interior, **el máximo y el mínimo de $u$ están en el borde**: si el máximo estuviera en un nodo interior, ese valor sería el promedio de sus cuatro vecinos, que no pueden ser mayores, así que los cuatro valen lo mismo; repitiendo el argumento, llegás al borde por un camino de nodos con el mismo valor máximo. Esto da un **chequeo gratis** de cualquier solver: `u.min()` y `u.max()` del interior tienen que caer dentro de $[\min g, \max g]$. Es una condición *necesaria*, no suficiente: sirve para detectar algunos errores (un signo cambiado) y no otros (una escala equivocada que deja todo dentro del rango). En la Tarea 1(d) se prueba con dos errores provocados.

### Tarea 1: Ensamblar el esquema de cinco puntos, resolver y verificar

Escribí las siguientes funciones (con `scipy.sparse`; nada de `numpy.linalg` ni de bucles sobre nodos para armar la matriz):

* `ensamblar(n)`: la matriz rala $A = I\otimes T + T\otimes I$ de $-\Delta_h$ ($N\times N$, $N = (n-1)^2$), en formato CSC. Usá `sp.diags` y `sp.kron`.
* `dato_borde(n)`: el array $G$ de $(n+1)\times(n+1)$ con $g = 1$ en el lado superior (fila $i = n$) y $0$ en los otros tres lados (en las dos esquinas de arriba, que el esquema no usa, poné $0.5$; el resto del array, cero).
* `rhs(n, G, f=0.0)`: el vector $\mathbf b$ de largo $N$ (con `f` un escalar o un array $(n+1)\times(n+1)$).
* `resolver(n, G, f=0.0)`: devuelve el array $u$ de $(n+1)\times(n+1)$ con el borde igual a $G$ y la solución de $A\mathbf u = \mathbf b$ en el interior (`spsolve`).
* `u_exacta(x, y, K)`: la suma parcial de $K$ términos (los impares $k < K$) de la serie de arriba, para arrays `x`, `y`.
* `chequeo_maximo(u, G)`: `True` si los valores interiores de `u` están en $[\min g, \max g]$ (con `g` los valores del borde sin las esquinas) y `False` si no.

Después:

**(a)** Para $n = 20$ verificá que `ensamblar(n)` es simétrica, que tiene a lo sumo $5$ no nulos por fila y que coincide (hasta $10^{-9}$ en norma máxima) con `-numerico.laplaciano_2d(n-1, 1/n)`.

**(b)** Resolvé para $n = 20, 40, 80$, guardá las soluciones en el diccionario `soluciones[n]`, verificá que coinciden con `numerico.poisson_cinco_puntos(0.0, G, 1/n)` y graficá un mapa de color de cada una (una figura de tres paneles, misma escala de color). En el centro del cuadrado la solución tiene que valer exactamente $1/4$: ¿por qué? (pensá en la simetría del problema).

**(c)** Calculá, para cada $n$, los tres errores de arriba respecto de `u_exacta` (con `K` grande, por ejemplo `K = 2000`) y guardalos en `err_central[n]`, `err_rms[n]`, `err_max[n]`. Imprimí una tabla con los errores y los órdenes observados $\log_2(e_n/e_{2n})$.

**(d)** Aplicá `chequeo_maximo` a las tres soluciones. Después provocá dos errores en la resolución de $n = 20$: (1) resolver con el signo cambiado, `spsolve(-A, b)`, y (2) olvidarte de dividir por $h^2$ en la parte del borde (en el lado superior, que es donde $g\neq0$: `b` sin el $1/h^2$ para esos nodos, con la misma $A$). ¿Los detecta el chequeo? ¿Qué otro chequeo detectaría el segundo error?

**Qué se espera.** (a) diferencia $0$ (exacta). (b) diferencia $\lesssim10^{-12}$ con `poisson_cinco_puntos`; en el mapa, $u$ pasa de $1$ arriba a $0$ abajo, con una capa límite fina pegada a los lados izquierdo y derecho cerca de las esquinas de arriba. (c) error central del orden de $10^{-3}$ para $n=20$ con orden observado $\approx 2.0$; error cuadrático medio con orden $\approx 1$; error máximo constante cerca de $7\times10^{-3}$. (d) los tres chequeos dan `True`; el error (1) se detecta ($u\le0$), el (2) no (todo queda en $[0, h^2]$): lo detecta el valor en el centro, que no es $0.25$, o el residuo $\|A\mathbf u - \mathbf b\|$ calculado con el $\mathbf b$ correcto.

In [ ]:
def ensamblar(n):
    """Matriz rala de -Delta_h (cinco puntos), (n-1)^2 x (n-1)^2, formato CSC."""
    # TODO: T = tridiag(-1, 2, -1)/h^2 con sp.diags; A = kron(I, T) + kron(T, I)
    pass

def dato_borde(n):
    """G (n+1, n+1): 1 en el lado superior (fila n), 0 en los otros lados, 0.5 en las esquinas de arriba."""
    # TODO
    pass

def rhs(n, G, f=0.0):
    """Lado derecho b (vector de largo (n-1)^2): f + (1/h^2) * (valores de G de los vecinos que están en el borde)."""
    # TODO
    pass

def resolver(n, G, f=0.0):
    """Solución u (n+1, n+1) del esquema de cinco puntos para -Delta u = f, u = G en el borde."""
    # TODO: u = G.copy(); u[1:-1, 1:-1] = spsolve(A, b).reshape(n-1, n-1)
    pass

def u_exacta(x, y, K=2000):
    """Suma parcial de la serie de Fourier (k impar < K) para g = 1 arriba, 0 en los otros lados."""
    # TODO (usá la forma que no se desborda)
    pass

def chequeo_maximo(u, G, tol=1e-12):
    """True si los valores interiores de u están en [min g, max g] (g: borde sin las esquinas)."""
    # TODO
    pass

# (a)
# TODO: simétrica, a lo sumo 5 no nulos por fila, coincide con -numerico.laplaciano_2d(n-1, 1/n)

# (b)
soluciones = {}    # TODO: soluciones[n] para n = 20, 40, 80
# TODO: comparar con numerico.poisson_cinco_puntos; figura de tres paneles con mapa_color (misma escala)

# (c)
err_central, err_rms, err_max = {}, {}, {}    # TODO: errores respecto de u_exacta para cada n
# TODO: tabla de errores y órdenes observados

# (d)
# TODO: chequeo_maximo en las tres soluciones; provocar los errores (1) y (2) en n = 20

In [ ]:
# Verificación
assert set(soluciones) == {20, 40, 80} and all(soluciones[n].shape == (n + 1, n + 1) for n in soluciones)
assert abs(ensamblar(20) - ensamblar(20).T).max() == 0 and ensamblar(20).nnz <= 5 * 19 ** 2
assert abs(ensamblar(20) + numerico.laplaciano_2d(19, 1 / 20)).max() < 1e-9, "ensamblar no coincide con -laplaciano_2d"
for n in (20, 40, 80):
    assert np.abs(soluciones[n] - numerico.poisson_cinco_puntos(0.0, dato_borde(n), 1 / n)).max() < 1e-9, f"n = {n}: no coincide con poisson_cinco_puntos"
    assert abs(soluciones[n][n // 2, n // 2] - 0.25) < 1e-10, "en el centro u debería valer 1/4 (simetría)"
assert abs(u_exacta(0.5, 0.5, K=2000) - 0.25) < 1e-6, "u_exacta(1/2, 1/2) debería ser 1/4"
o1, o2 = np.log2(err_central[20] / err_central[40]), np.log2(err_central[40] / err_central[80])
assert 1.8 < o1 < 2.3 and 1.8 < o2 < 2.3, f"orden observado {o1:.2f}, {o2:.2f}: se esperaba ~2"
assert err_central[20] < 3e-3 and 5e-3 < err_max[80] < 1e-2, "errores fuera de lo esperado"
assert chequeo_maximo(soluciones[20], dato_borde(20)) and not chequeo_maximo(-soluciones[20], dato_borde(20))
print("cinco puntos: OK")

## 2. Cuánto cuesta: matriz densa contra matriz rala

El sistema tiene $N = (n-1)^2 \approx n^2$ incógnitas. Hay dos maneras de guardar $A$ y de resolverla:

* **Densa** (`A.toarray()` y `np.linalg.solve`): la matriz guarda $N^2 \approx n^4$ números (8 bytes cada uno) y la eliminación gaussiana (o Cholesky) cuesta $\sim N^3 \approx n^6$ operaciones. Duplicar $n$ multiplica la memoria por $16$ y el tiempo por $64$.
* **Rala** (`scipy.sparse`, `spsolve`): $A$ tiene $\approx 5N$ no nulos, así que guardarla cuesta $O(n^2)$: las tres listas del formato CSC son `data` (los valores), `indices` (las filas) e `indptr` (dónde empieza cada columna), y su tamaño total es `A.data.nbytes + A.indices.nbytes + A.indptr.nbytes`. `spsolve` factoriza $A = LU$ reordenando las incógnitas para que $L$ y $U$ tengan pocos no nulos (el "**fill-in**"): con un buen reordenamiento (*nested dissection*) el costo para la grilla 2D es $O(N^{3/2}) = O(n^3)$ en tiempo y $O(N\log N)$ en memoria (con el reordenamiento por defecto de SuperLU, `COLAMD`, es algo peor pero del mismo orden). Podés ver el fill-in directamente con `lu = spla.splu(A)`: la cantidad de no nulos de los factores es `lu.L.nnz + lu.U.nnz`. Otra ventaja de factorizar una sola vez con `splu`: `lu.solve(b)` resuelve luego cualquier lado derecho nuevo casi gratis (útil cuando el dato cambia y la matriz no).

**Cómo medir.** El tiempo con `time.perf_counter()` (para tamaños chicos repetí la medición y tomá el mínimo: los milisegundos son ruidosos); la memoria de la matriz densa con `Ad.nbytes`, la de la rala con la suma de arriba. Para ver el **exponente** graficá en escala log-log contra $n$ y ajustá una recta con `np.polyfit(np.log(n), np.log(t), 1)`: la pendiente es el exponente. **Qué esperar:** memoria densa con pendiente $\approx4$ y rala $\approx2$ (un poco menos, porque $N = (n-1)^2$ y no $n^2$); tiempo denso con pendiente entre $4.5$ y $6$ (la teoría dice $6$, pero con tamaños chicos manda el costo de la librería optimizada, y aparecen efectos de caché) y tiempo ralo con pendiente entre $2$ y $3$. Los números absolutos dependen de la computadora; **los exponentes no**.

### Tarea 2: Tiempo y memoria: denso contra ralo

Con `ensamblar(n)` de la Tarea 1 y el lado derecho `rhs(n, dato_borde(n))`:

**(a)** Para $n \in \{10, 20, 30, 40, 60, 80\}$ medí el tiempo de resolver con matriz densa (`np.linalg.solve(A.toarray(), b)`) y la memoria de la matriz densa. Guardalos en los arrays `t_denso` y `mem_denso` (bytes). No pases de $n = 80$ (la matriz ya ocupa $300$ MB).

**(b)** Para $n \in \{10, 20, 40, 80, 160, 320, 480\}$ medí el tiempo de `spsolve`, la memoria de la matriz rala (en bytes) y el fill-in (`nnz` de $L$ más $U$ por `splu`, dividido por `A.nnz`). Guardalos en `t_ralo`, `mem_ralo` y `fill`.

**(c)** Graficá en log-log el tiempo y la memoria contra $n$ (dos paneles, denso y ralo en cada uno) y ajustá los exponentes: `exp_t_denso`, `exp_t_ralo`, `exp_mem_denso`, `exp_mem_ralo` (para el tiempo ajustá solo con los $n\ge 40$: los tiempos de $n$ chico son ruido).

**(d)** Con los exponentes ajustados (y la memoria a $n = 80$ como referencia) estimá: ¿qué $n$ máximo se puede resolver con matriz densa en $16$ GB de memoria? ¿Y en cuánto tiempo se resolvería $n = 1000$ (un millón de incógnitas) con matriz rala, si la memoria no fuera problema? Escribí las cuentas.

**Qué se espera.** Exponentes de memoria $\approx 4$ (denso) y $\approx 2$ (ralo); de tiempo entre $4.5$ y $6$ (denso) y entre $2$ y $3$ (ralo). A $n=80$ el denso pesa $\approx 300$ MB y el ralo $\approx 0.4$ MB (un factor $\approx 800$); el fill-in crece lentamente con $n$ (de $\approx3$ veces los no nulos de $A$ en $n=10$ a $\approx24$ en $n=480$: $L$ y $U$ tienen más no nulos que $A$, pero siguen siendo $O(N\log N)$). El $n$ máximo denso es del orden de $200$; el ralo hace $n = 1000$ en unas decenas de segundos.

In [ ]:
ns_d = [10, 20, 30, 40, 60, 80]
ns_r = [10, 20, 40, 80, 160, 320, 480]
t_denso, mem_denso = [], []          # TODO: tiempo (s) y memoria (bytes) de la resolución densa para cada n en ns_d
t_ralo, mem_ralo, fill = [], [], []  # TODO: tiempo, memoria (bytes) y fill-in para cada n en ns_r

# TODO: figura log-log (tiempo y memoria contra n), ajuste de exponentes
exp_t_denso = exp_t_ralo = exp_mem_denso = exp_mem_ralo = None   # TODO (np.polyfit sobre log n, log t)

# TODO (d): n máximo denso en 16 GB; tiempo estimado del ralo para n = 1000

In [ ]:
# Verificación
assert abs(exp_mem_denso - 4) < 0.3 and abs(exp_mem_ralo - 2) < 0.15, "los exponentes de memoria deberían ser 4 y 2"
assert 4.0 < exp_t_denso < 6.5, f"exponente de tiempo denso {exp_t_denso:.2f}: se esperaba entre 4.5 y 6"
assert 1.8 < exp_t_ralo < 3.3, f"exponente de tiempo ralo {exp_t_ralo:.2f}: se esperaba entre 2 y 3"
assert mem_denso[-1] / mem_ralo[ns_r.index(80)] > 300, "a n = 80 el denso debería pesar cientos de veces más que el ralo"
print("costo denso/ralo: OK")

## 3. Resolver jugando: Monte Carlo

### El juego y el estimador

Desde un nodo interior $p = (i,j)$ el jugador se mueve a uno de sus cuatro vecinos con probabilidad $\tfrac14$, y repite hasta que toca un nodo del borde $q$; ahí cobra $g(q)$. Sea $u(p)$ el **valor esperado del pago** cuando se parte de $p$. Condicionando en el primer paso (cada vecino con probabilidad $\frac14$, y desde el vecino el juego empieza de nuevo) resulta

$$u(p) = \frac14\sum_{p'\sim p} u(p'),\qquad u(q) = g(q)\ \ (q\in\text{borde}),$$

que es **exactamente** la ecuación de promedios del esquema de cinco puntos. Como el sistema lineal tiene solución única, el valor esperado del juego es la solución del esquema de cinco puntos, *sin ningún error de discretización aparte* (el error de discretización, $O(h^2)$, es el del esquema respecto de la ecuación continua, y es igual en los dos métodos).

Para estimar $u(p)$ se simulan $M$ partidas independientes, cada una con pago $X_1,\dots,X_M$, y se promedia:

$$\hat u_M(p) = \frac1M\sum_{m=1}^M X_m,\qquad \mathbb E[\hat u_M] = u(p),\qquad \operatorname{sd}(\hat u_M) = \frac{\sigma(p)}{\sqrt M},\quad \sigma^2 = \operatorname{Var}(X).$$

El estimador no tiene sesgo y su **error estadístico cae como $M^{-1/2}$** (ley de los grandes números y teorema central del límite): para dividir el error por $10$ hay que simular $100$ veces más partidas. La $\sigma$ se estima con el desvío muestral de los pagos, y $\hat\sigma/\sqrt M$ (el "error estándar") es el error de la estimación *que podés calcular sin conocer la solución*. Cuando el pago es $0$ o $1$ (como en nuestro problema, con $g\in\{0,1\}$) $X$ es una Bernoulli de parámetro $u$ y $\sigma = \sqrt{u(1-u)}\le\frac12$: $u$ es la **probabilidad** de salir por el lado de arriba.

### Cuánto cuesta una partida

Un paseo al azar tarda $\sim n^2$ pasos en llegar al borde (para desplazarse una distancia $L$ a saltos de tamaño $h$ hacen falta $\sim (L/h)^2$ pasos): en el cuadrado, desde el centro, el número esperado de pasos es $\approx 0.29\,n^2$ (lo vas a medir en la Tarea 4 y a explicar en la Tarea 6). Entonces:

* estimar **un** valor $u(p)$ con error estadístico $\varepsilon$ cuesta $M = (\sigma/\varepsilon)^2$ partidas de $\approx 0.29\,n^2$ pasos cada una: $\sim \sigma^2 n^2/\varepsilon^2$ pasos;
* estimar **toda** la función en los $N\approx n^2$ nodos cuesta $N$ veces eso, $\sim n^4/\varepsilon^2$; y para que el error *total* sea comparable con el de discretización ($\sim h^2 = n^{-2}$), $\varepsilon\sim n^{-2}$: $\sim n^{8}$. Mientras que el sistema ralo cuesta $\sim n^3$ y da **todos** los valores con precisión de máquina.

Entonces, en 2D, **el sistema lineal ralo le gana por muchísimo a Monte Carlo**. ¿Para qué sirve Monte Carlo, entonces? Para dos cosas: si sólo querés el valor en *un* punto (con un error modesto, digamos $10^{-2}$), y sobre todo si la dimensión es alta. La grilla en dimensión $d$ tiene $N = n^d$ nodos: con $n = 20$ y $d = 10$, $N \approx 10^{13}$ y ni siquiera podés guardar un vector con una incógnita por nodo. Pero el juego en $d$ dimensiones es igual de simple (un paso es elegir una de las $2d$ direcciones), el **error estadístico $\sigma/\sqrt M$ no depende de $d$** y el costo por partida crece apenas como una potencia de $d$. Ese es el sentido en que Monte Carlo "no sufre la maldición de la dimensión". En la Tarea 4 lo vas a ver con un problema de dimensión $d$ cuya respuesta exacta se conoce por simetría.

**Cómo programarlo.** Simular una partida por vez con un bucle de Python es lento; conviene simular **las $M$ partidas a la vez** con arrays de `numpy`: un vector con la posición (`ii`, `jj`) de cada jugador, otro con quién sigue jugando, y en cada paso se mueven sólo los que siguen activos (un jugador que llega al borde cobra y sale). El número de iteraciones del bucle `while` es el del paseo más largo, unos cientos de pasos. Contá también el **costo** en "pasos de jugador" (suma de los largos de las partidas): es una medida del trabajo que no depende de la computadora.

### Tarea 3: Monte Carlo: `valor_mc`, mapas y error contra $M$

Con el mismo problema de la Tarea 1 ($g=1$ arriba, $0$ en los otros lados), $n = 20$ y la solución de cinco puntos `soluciones[20]` como referencia:

**(a)** Escribí `valor_mc(i, j, M, G, rng=None)`: simula $M$ partidas a la vez desde el nodo interior $(i,j)$ (los cuatro movimientos con igual probabilidad, hasta tocar el borde) y devuelve la tripla `(media, error_estandar, pasos)` con la media de los pagos, el error estándar `std(ddof=1)/sqrt(M)` y el total de pasos de jugador (suma de los largos de las $M$ partidas). Usá un generador `rng = np.random.default_rng(semilla)` para que los resultados se puedan repetir.

**(b)** Hacé un mapa de $u$ por Monte Carlo (en los $19^2$ nodos interiores) para $M = 10, 100, 1000$ y compará con el de cinco puntos: una figura de cuatro paneles (los tres MC y la referencia, misma escala). Imprimí, para cada $M$, el error máximo y el error cuadrático medio contra la referencia, y guardá los mapas en `U_mc[M]` (arrays $(n+1)\times(n+1)$ con el borde igual a $G$).

**(c)** *El error contra $M$.* En el nodo $(i,j) = (15, 10)$ (o sea $(x,y) = (0.5, 0.75)$) medí el error cuadrático medio de $\hat u_M$ respecto de la referencia, con $M \in \{10, 30, 100, 300, 1000, 3000, 10^4\}$, repitiendo la estimación $R = 100$ veces con semillas distintas para cada $M$ (el error de *una* estimación es una variable aleatoria: lo que tiene sentido graficar es su desvío típico). Guardá los errores en `rms[M]`. Graficá en log-log `rms` contra $M$ junto con la recta teórica $\sigma/\sqrt M$, $\sigma = \sqrt{u(1-u)}$, y ajustá la pendiente (`pend`).

**(d)** Con la misma corrida de (c) medí el **costo** de una llamada en función de $M$ (pasos de jugador y tiempo) y comprobá que crece linealmente con $M$; calculá los pasos por partida.

**Qué se espera.** (b) los tres mapas son cada vez menos ruidosos; con $M=10$ el error máximo es del orden de $0.3$, con $M=100$ de $0.1$, con $M=1000$ de $0.04$ (el máximo sobre $361$ nodos crece por encima del típico; el error cuadrático medio baja como $M^{-1/2}$). (c) pendiente $-0.5\pm0.1$, con la curva medida sobre la teórica (el $\sigma$ del pago es $\approx 0.5$); $u(15,10)\approx0.54$. (d) los pasos de jugador crecen linealmente con $M$ y cada partida dura del orden de $n^2$ pasos (unos $90$ desde ese nodo, que está más cerca del borde que el centro; lo que importa es que sea $O(n^2)$ y que no dependa de $M$); el tiempo de pared crece más lento que $M$ para $M$ chico, porque cada iteración del bucle tiene un costo fijo de Python.

In [ ]:
def valor_mc(i, j, M, G, rng=None):
    """Simula M partidas desde el nodo interior (i, j) con pago G en el borde.
    Devuelve (media, error_estandar, pasos_totales)."""
    # TODO: todas las partidas a la vez con arrays de numpy (posiciones ii, jj; activo; pago)
    pass

n = 20
G = dato_borde(n)
u_ref = soluciones[20]
rng = np.random.default_rng(0)

# (b)
U_mc = {}     # TODO: U_mc[M] para M = 10, 100, 1000 (mapa sobre los nodos interiores)
# TODO: figura de cuatro paneles; imprimir error máximo y error cuadrático medio de cada M

# (c)
Ms = [10, 30, 100, 300, 1000, 3000, 10000]
rms = {}      # TODO: rms[M] = sqrt(mean((estimaciones - u_ref[15, 10])**2)) sobre R = 100 repeticiones
pend = None   # TODO: pendiente del ajuste log-log
# TODO: figura log-log con la recta teórica sigma / sqrt(M)

# (d)
# TODO: pasos y tiempo en función de M; pasos por partida

In [ ]:
# Verificación
m_, e_, p_ = valor_mc(10, 10, 4000, dato_borde(20), np.random.default_rng(5))
assert abs(m_ - 0.25) < 4 * e_ + 1e-12 and p_ > 4000, f"valor_mc en el centro: {m_:.3f} +- {e_:.3f} (debería ser 0.25)"
assert set(U_mc) == {10, 100, 1000}
e = [np.sqrt(np.mean((U_mc[M] - u_ref)[1:-1, 1:-1] ** 2)) for M in (10, 100, 1000)]
assert e[0] > e[1] > e[2] and 0.5 < e[0] / e[2] / 10 < 2, f"errores {e}: el cociente entre M = 10 y M = 1000 debería ser ~10"
assert -0.62 < pend < -0.38, f"pendiente {pend:.2f}: se esperaba -0.5"
print("Monte Carlo: OK")

### Comparar costos, y qué pasa en dimensión alta

Las dos tareas que siguen ponen números a la discusión de arriba. Primero, **cuánto tarda un paseo**: si el paseo dura $\sim c\,n^2$ pasos, estimar un valor con error $\varepsilon$ cuesta $\sim(\sigma/\varepsilon)^2\,c\,n^2$ pasos, y hay que ver cuánto vale $c$. Segundo, **cuánto cuestan los dos métodos para la misma respuesta**: tomamos un solo nodo, exigimos un error estadístico de $\varepsilon = 0.01$ (dos decimales) y comparamos el tiempo de Monte Carlo *en ese nodo* con el tiempo del sistema ralo, que da *todos los nodos* con error de máquina. Tercero, la **dimensión**: en el cubo $[0,n]^d$ (con $n = 20$), un jugador que parte del centro elige en cada paso una de las $d$ coordenadas y un signo, y termina al tocar una cara. Si el pago es $1$ en la cara $x_d = n$ y $0$ en las otras $2d-1$, por simetría el valor esperado en el centro es exactamente $\frac{1}{2d}$ (las $2d$ caras son equiprobables): tenemos la respuesta exacta en cualquier dimensión, y podemos ver el error de Monte Carlo sin que haga falta ninguna grilla.

### Tarea 4: Costo de Monte Carlo contra cinco puntos, y dimensión

Usá `valor_mc` (Tarea 3) y `resolver` (Tarea 1).

**(a)** *Duración de un paseo.* Para $n = 10, 20, 40, 80$ estimá el número medio de pasos que dura una partida que sale del centro del cuadrado ($M = 400$ partidas; usá el tercer valor que devuelve `valor_mc`, dividido por $M$). Ajustá el exponente en $n$ (`exp_T`) y el coeficiente $c = \bar T / n^2$ (`c_T`).

**(b)** *Mismo error, dos costos.* Para $n = 20, 40, 80$ y el nodo $(i,j) = (3n/4,\ n/2)$ (o sea $(x,y) = (0.5, 0.75)$): tomá $\sigma = \sqrt{u(1-u)}$ con $u$ la solución de cinco puntos en ese nodo, elegí $M = \lceil \sigma^2/\varepsilon^2\rceil$ para $\varepsilon = 0.01$ y medí: el tiempo de `resolver(n, G)` (todos los nodos, error de máquina respecto del sistema), el tiempo de `valor_mc` con ese $M$ (un nodo, error estadístico $\approx 0.01$) y los pasos de jugador. Armá una tabla con esos tiempos, el cociente y el tiempo *estimado* de Monte Carlo para todos los nodos, $(n-1)^2$ veces el de un nodo. Verificá que el valor de Monte Carlo cae a menos de $3$ errores estándar del de cinco puntos.

**(c)** *Dimensión.* Escribí `valor_mc_dim(d, n, M, rng)`: $M$ partidas a la vez en el cubo $\{0,\dots,n\}^d$, desde el centro $(n/2,\dots,n/2)$; en cada paso se elige una coordenada al azar y un signo $\pm1$, y el jugador termina cuando esa coordenada llega a $0$ o a $n$; el pago es $1$ si terminó en $x_d = n$ y $0$ en cualquier otro caso. Devuelve `(media, error_estandar, pasos)`. Para $d = 2, 3, 5, 10, 20$, $n = 20$ y $M = 4000$, armá una tabla con: el valor exacto $\frac1{2d}$, el estimado, el error estándar, el error real dividido por el error estándar, los pasos por partida, la cantidad de nodos de la grilla $N = (n-1)^d$ y la memoria de un solo vector de $N$ números (en GB, 8 bytes por número).

**Qué se espera.** (a) exponente $\approx 2$ y $c\approx 0.29$ (en la Tarea 6 se explica de dónde sale). (b) para $n = 20$ el sistema ralo tarda un par de milisegundos y Monte Carlo (un solo nodo, $M\approx 2500$) unas decenas de milisegundos: ya para **un** nodo es entre $10$ y $20$ veces más caro, aunque el ralo resuelva todos; Monte Carlo para *todos* los nodos es de miles a cientos de miles de veces más lento (segundos en $n=20$, media hora en $n=80$). (c) el estimado coincide con $\frac{1}{2d}$ dentro de un par de errores estándar en todas las dimensiones; el error estándar es $\sqrt{p(1-p)/M}$ con $p = \frac{1}{2d}$, es decir **menor** en dimensión alta (porque $p$ es chica) y desde luego no crece; los pasos por partida crecen con $d$, pero muy lento (de $\approx 120$ en $d=2$ a $\approx 360$ en $d=20$); la memoria de la grilla pasa de $\approx 3$ KB en $d=2$ a $\approx 10^{14}$ TB en $d = 20$.

In [ ]:
# (a)
exp_T, c_T = None, None    # TODO: pasos medios desde el centro para n = 10, 20, 40, 80 (M = 400); exponente y coeficiente
# TODO: imprimir la tabla

# (b)
epsilon = 0.01
# TODO: para n = 20, 40, 80: sigma, M, tiempo de resolver, tiempo de valor_mc, pasos, cociente, estimación para todos los nodos

# (c)
def valor_mc_dim(d, n, M, rng=None):
    """M partidas en el cubo {0..n}^d desde el centro; pago 1 si termina en x_d = n. Devuelve (media, error_estandar, pasos)."""
    # TODO
    pass

# TODO: tabla para d = 2, 3, 5, 10, 20 (n = 20, M = 4000)

In [ ]:
# Verificación
assert 1.85 < exp_T < 2.15 and 0.25 < c_T < 0.34, f"exponente {exp_T:.2f}, c = {c_T:.3f}: se esperaba ~2 y ~0.29"
m2, e2, p2 = valor_mc_dim(2, 20, 4000, np.random.default_rng(1))
assert abs(m2 - 0.25) < 4 * e2, f"d = 2: {m2:.3f} +- {e2:.3f}, debería ser 0.25"
for d in (3, 5, 10):
    md_, ed_, _ = valor_mc_dim(d, 20, 4000, np.random.default_rng(10 + d))
    assert abs(md_ - 1 / (2 * d)) < 4 * ed_, f"d = {d}: {md_:.4f} +- {ed_:.4f}, debería ser {1 / (2 * d):.4f}"
assert valor_mc_dim(10, 20, 500, np.random.default_rng(0))[2] / 500 > valor_mc_dim(2, 20, 500, np.random.default_rng(0))[2] / 500
print("costos y dimensión: OK")

## 4. "Jugar a promediar": Jacobi, Gauss–Seidel y gradiente conjugado

La ecuación de promedios sugiere un algoritmo trivial: **empezar con cualquier cosa y promediar una y otra vez**. En **Jacobi** todos los nodos interiores se actualizan a la vez con los valores de la iteración anterior:

$$u^{k+1}_{i,j} = \frac14\bigl(u^k_{i+1,j} + u^k_{i-1,j} + u^k_{i,j+1} + u^k_{i,j-1}\bigr)\ \ \Bigl(+\tfrac{h^2}{4}f_{i,j}\Bigr),$$

con el borde fijo en $g$. En forma matricial, con $D = \frac{4}{h^2}I$ la diagonal de $A$: $\mathbf u^{k+1} = \mathbf u^k + D^{-1}(\mathbf b - A\mathbf u^k)$: cada iteración corrige la solución con el residuo. En **Gauss–Seidel** cada nodo usa los valores más nuevos que ya se calcularon en esta misma pasada. Con el recorrido lexicográfico (por filas) es un bucle de Python inevitable; con el **orden rojo–negro** (los nodos con $i+j$ par —rojos— y los de $i+j$ impar —negros—: los vecinos de un rojo son todos negros y viceversa) se actualizan primero todos los rojos, a la vez (con `numpy`), y después todos los negros usando los rojos ya nuevos: una "iteración" es esa pasada doble, y se vectoriza igual que Jacobi.

**Por qué convergen y por qué son lentas.** El error $\mathbf e^k = \mathbf u^k - \mathbf u$ cumple $\mathbf e^{k+1} = B\,\mathbf e^k$ con $B = I - D^{-1}A$, así que $\|\mathbf e^k\|\sim\rho(B)^k$, con $\rho(B)$ el radio espectral. Para el cuadrado los autovalores de $B$ son $\frac12(\cos\frac{p\pi}{n} + \cos\frac{q\pi}{n})$, $1\le p,q\le n-1$, y el mayor es

$$\rho_J = \cos\frac{\pi}{n} \approx 1 - \frac{\pi^2}{2n^2}.$$

Está pegadísimo a $1$, y más cuanto más fina es la grilla: para reducir el error por un factor $\text{tol}$ hacen falta

$$k \approx \frac{\ln(1/\text{tol})}{-\ln\rho_J} \approx \frac{2\ln(1/\text{tol})}{\pi^2}\,n^2 \approx 1.4\,n^2\quad(\text{tol}=10^{-3})$$

iteraciones, y **cada iteración cuesta $O(N) = O(n^2)$**: el costo total de Jacobi es $O(n^4)$, peor que el del sistema ralo directo ($O(n^3)$). Gauss–Seidel tiene $\rho_{GS} = \rho_J^2$ (para este ordenamiento): la mitad de las iteraciones, con la misma pendiente $n^2$. Hay una lectura probabilística muy linda: si empezás con $u^0 = 0$ en el interior, $u^k(p)$ es el pago esperado del juego *contando sólo las partidas que ya terminaron en a lo sumo $k$ pasos* ($\mathbb E[g(X_T);\,T\le k]$). El error es lo que cobrarían las partidas que todavía no llegaron al borde, y como un paseo tarda $\sim n^2$ pasos en llegar, hacen falta $\sim n^2$ iteraciones: **la información viaja una celda por iteración, y el borde está a $n$ celdas, pero como difunde (no viaja en línea recta) tarda $n^2$**.

**Gradiente conjugado.** Como $A$ es simétrica y definida positiva, resolver $A\mathbf u = \mathbf b$ es minimizar la energía $\frac12\mathbf u^TA\mathbf u - \mathbf b^T\mathbf u$ (el análogo discreto de $\frac12\int|\nabla u|^2 - \int fu$: la formulación variacional de Dirichlet). El gradiente conjugado (`scipy.sparse.linalg.cg`) minimiza esa energía sobre subespacios de Krylov y converge en $\sim\sqrt{\kappa(A)}$ iteraciones, con $\kappa(A)\sim n^2$ el número de condición: **$\sim n$ iteraciones**, contra las $n^2$ de Jacobi. Con precondicionadores (multigrilla, por ejemplo) se llega a $O(1)$ iteraciones y costo $O(N)$ total: es el estado del arte para problemas grandes. Es una lectura: acá sólo vamos a contar las iteraciones de CG para ver la pendiente $n^1$.

**Criterio de parada.** No tiene sentido iterar hasta el error de máquina: la solución del sistema ya tiene un error $O(h^2)$ respecto de la ecuación continua (en $n=20$ es $\approx10^{-3}$, Tarea 1). Acá usamos como criterio $\max|u^k - u_h| < 10^{-3}$, midiendo el error contra la solución **del sistema** (que sabemos calcular); en la práctica no se conoce y se usa el residuo $\|\mathbf b - A\mathbf u^k\|$.

### Tarea 5: Jacobi y Gauss–Seidel: cuántas iteraciones hacen falta

Con el mismo problema ($g = 1$ arriba, $0$ en los otros lados) y `soluciones[n]` (Tarea 1) como referencia:

**(a)** Escribí `jacobi(G, tol, u_ref)` y `gauss_seidel_rn(G, tol, u_ref)` (rojo–negro): parten de $u = 0$ en el interior y $u = G$ en el borde, iteran hasta que $\max|u - u_{ref}| < $ `tol` y devuelven `(k, u, hist)` con el número de iteraciones, la solución final y la lista `hist` del error máximo en cada iteración (empezando por la iteración $0$). Para Jacobi, la actualización con *slices* de `numpy` (`u[1:-1,1:-1] = (u[2:,1:-1] + ...)/4`) ya es Jacobi, porque `numpy` calcula el lado derecho completo antes de asignar.

**(b)** Para $n = 20, 40, 80$ y `tol = 1e-3` medí las iteraciones de cada método (`iter_J[n]`, `iter_GS[n]`), imprimí una tabla con el cociente $\text{GS}/\text{J}$, el cociente entre $n$ consecutivos y la predicción $\ln(\text{tol})/\ln\cos(\pi/n)$, y ajustá el exponente de $n$ (`exp_J`, `exp_GS`).

**(c)** Para $n = 40$ graficá en escala semilogarítmica el error contra la iteración para Jacobi, Gauss–Seidel y (después de (d)) gradiente conjugado, y verificá que las curvas de Jacobi y Gauss–Seidel son rectas (convergencia geométrica) con pendientes en cociente $\approx 2$.

**(d)** *Gradiente conjugado.* Con `spla.cg(A, b, callback=cb, maxiter=5000)` (la `A` y el `b` de la Tarea 1) registrá el error máximo contra la referencia en cada iteración (el `callback` recibe el vector $\mathbf x_k$ de las incógnitas) y contá las iteraciones hasta bajar de `tol`, para $n = 20, 40, 80, 160$ (`iter_CG[n]`; para $n=160$ la referencia se calcula con `resolver`). Ajustá el exponente (`exp_CG`).

**Qué se espera.** (b) Jacobi: unas $485$, $1945$ y $7786$ iteraciones (cociente $\approx 4$ entre $n$ consecutivos: exponente $2$); Gauss–Seidel, la mitad; la predicción con $\rho_J$ es un poco mayor que lo observado (porque el error inicial es menor que $1$ y $\text{tol}$ se mide en norma máxima). (c) rectas (después de un transitorio de pocas iteraciones) con Gauss–Seidel de pendiente doble; la de Jacobi es $\ln\cos(\pi/n)\approx-0.0031$ por iteración en $n=40$; la curva de CG no es una recta. (d) $\approx 31$, $62$, $126$, $252$ iteraciones (se duplican con $n$): exponente $\approx1$. Conclusión: para lograr el mismo error, Jacobi cuesta $O(n^2)\times O(n^2) = O(n^4)$, CG $O(n)\times O(n^2) = O(n^3)$, el directo ralo $O(n^3)$ con una constante mucho mayor de memoria.

In [ ]:
def jacobi(G, tol, u_ref, maxit=200000):
    """Jacobi desde u = 0 en el interior. Devuelve (k, u, hist)."""
    # TODO
    pass

def gauss_seidel_rn(G, tol, u_ref, maxit=200000):
    """Gauss-Seidel rojo-negro desde u = 0 en el interior. Devuelve (k, u, hist)."""
    # TODO
    pass

# (b)
iter_J, iter_GS = {}, {}      # TODO: iteraciones para n = 20, 40, 80 con tol = 1e-3
exp_J = exp_GS = None         # TODO: exponente de n (ajuste log-log)
# TODO: tabla (cociente GS/J, cociente entre n consecutivos, predicción con cos(pi/n))

# (c)
# TODO: figura semilogy del error contra la iteración para n = 40 (Jacobi, GS y, después de (d), CG)

# (d)
iter_CG = {}                  # TODO: iteraciones de CG hasta error < 1e-3 para n = 20, 40, 80, 160
exp_CG = None                 # TODO

In [ ]:
# Verificación
assert 350 < iter_J[20] < 650 and 3.6 < iter_J[40] / iter_J[20] < 4.4 and 3.6 < iter_J[80] / iter_J[40] < 4.4, f"Jacobi: {iter_J}"
assert all(0.45 < iter_GS[n] / iter_J[n] < 0.56 for n in (20, 40, 80)), f"GS / Jacobi: {[iter_GS[n] / iter_J[n] for n in (20, 40, 80)]}, se esperaba ~1/2"
assert 1.85 < exp_J < 2.15 and 0.8 < exp_CG < 1.25, f"exponentes: Jacobi {exp_J:.2f}, CG {exp_CG:.2f}"
u_j = jacobi(dato_borde(20), 1e-8, soluciones[20])[1]
assert np.abs(u_j - soluciones[20]).max() < 1e-7, "Jacobi debería converger a la solución del sistema"
print("iterativos: OK")

## 5. Con fuente: $-\Delta u = f$ y el juego con pago acumulado

Con una fuente, el esquema de cinco puntos es $-\Delta_h u = f$, es decir

$$u_{i,j} = \frac14\bigl(u_{i+1,j}+u_{i-1,j}+u_{i,j+1}+u_{i,j-1}\bigr) + \frac{h^2}{4}\,f_{i,j}.$$

**Cómo cambia el juego.** Ahora el jugador, además del pago final $g$ en el borde, **cobra $\frac{h^2}{4}f(p)$ cada vez que está parado en un nodo interior $p$** (contando el nodo de partida, y sin contar el del borde donde termina). Si $X_0 = p, X_1, \dots, X_T$ es el paseo ($X_T$ en el borde),

$$u(p) = \mathbb E\Bigl[\,g(X_T) + \frac{h^2}{4}\sum_{k=0}^{T-1} f(X_k)\Bigr].$$

*Por qué:* condicionando en el primer paso, $u(p) = \frac{h^2}{4}f(p) + \frac14\sum_{p'\sim p}u(p')$ (cobrás la fuente del nodo donde estás y seguís jugando desde el vecino al que fuiste), que es la ecuación de arriba. Es la versión discreta de la **fórmula de Feynman–Kac**: $u(x) = \mathbb E\bigl[g(B_\tau) + \int_0^\tau f(B_s)\,ds\bigr]$ para un movimiento browniano $B$ con generador $\Delta$, que sale del dominio en el instante aleatorio $\tau$. **El factor $h^2/4$** es lo que dura cada paso: para que el paseo con pasos de largo $h$ aproxime a un browniano con generador $\Delta$ (varianza $2t$ por coordenada, o sea $\mathbb E|X_t - X_0|^2 = 4t$ en el plano), cada paso tiene que durar $\tau_h = h^2/4$, y la integral $\int f\,ds$ se aproxima por $\sum f(X_k)\,\tau_h$.

Un caso particular que da la clave del costo. Con $g = 0$ y $f = 1$ resulta $u(p) = \frac{h^2}{4}\,\mathbb E[T]$: **la solución de $-\Delta u = 1$ con $u=0$ en el borde es (salvo el factor $h^2/4$) el número esperado de pasos hasta el borde**. En el centro del cuadrado unitario la solución de $-\Delta u = 1$ vale $\approx 0.0737$, de donde $\mathbb E[T] = 4u/h^2 \approx 0.295\,n^2$: es el "$0.29\,n^2$" que midieron en la Tarea 4.

**Principio del máximo con fuente.** Si $-\Delta u = f\ge0$ ($u$ superarmónica) el **mínimo** de $u$ está en el borde (y $u\ge\min g$), pero el **máximo puede estar adentro**: con $f=1$, $g=0$ la solución es positiva en todo el cuadrado y vale $0$ en el borde. Además vale el **principio de comparación**: si $f\ge0$, la solución con fuente es mayor o igual que la solución sin fuente (con el mismo $g$) en todo el cuadrado: sumás un pago extra no negativo al juego. Con $f\le0$ (subarmónica) es el máximo el que queda en el borde.

### Tarea 6: Con fuente: cinco puntos contra Monte Carlo con pago acumulado

**(a)** Escribí `valor_mc_f(i, j, M, G, F, rng=None)`: como `valor_mc` pero el jugador acumula $\frac{h^2}{4}F[i', j']$ en cada nodo interior $(i',j')$ en que está parado (incluyendo el de partida), con $h = 1/n$ y `F` un array $(n+1)\times(n+1)$ con los valores de $f$ en los nodos. Devuelve `(media, error_estandar, pasos)` con la media del pago total (final más acumulado).

**(b)** Caso $f = 1$, $g = 0$ ($n = 20$, `G = np.zeros((21, 21))`, `F = np.ones((21, 21))`). Compará Monte Carlo ($M = 20000$) con `resolver(20, G, F)` en los nodos $(10,10)$, $(5,5)$, $(10,3)$ y $(15,10)$: una tabla con el valor de cinco puntos, el de Monte Carlo, el error estándar y $z = (\text{MC} - \text{5p})/\text{error estándar}$. Con $n = 80$ y `resolver`, el valor en el centro aproxima el de la ecuación continua, $0.07367$: verificalo.

**(c)** *El tiempo de salida.* Con los mismos datos, `pasos / M` que devuelve `valor_mc_f` desde el centro es el número medio de pasos $\mathbb E[T]$. Verificá la relación $u_h(\text{centro}) = \frac{h^2}{4}\,\mathbb E[T]$ comparando $\frac{4}{h^2}\,u_h$ con `pasos / M`, y explicá por qué el $0.29\,n^2$ de la Tarea 4 sale de acá.

**(d)** Una fuente que no es constante ni un dato nulo: $f(x,y) = 40\,x$ y $g$ el dato de la Tarea 1 (`dato_borde(20)`). Repetí la comparación de (b) en los mismos nodos y verificá $|z| < 4$ en todos.

**(e)** *Principio del máximo.* Con `resolver` en $n = 40$ y $g=0$, calculá la solución para $f = +1$ y $f = -1$ y verificá: con $f = 1$ el mínimo interior es $\ge 0$ pero el máximo es $>0 = \max g$ (¿dónde está?); con $f=-1$ pasa lo simétrico. Con $g$ = `dato_borde(40)` y $f = 50\,\sin(\pi x)\sin(\pi y)\ge0$ verificá el principio de comparación: `u_con_fuente >= u_sin_fuente` en todos los nodos.

**Qué se espera.** (b) $z$ entre $-3$ y $3$ en los cuatro nodos (con $M = 20000$ el error estándar es del orden de $10^{-3}$, sobre valores de $0.04$ a $0.07$; como cada partida cobra sobre $\approx 100$ pasos, la desviación del pago total es del orden del propio valor); $u_h(0.5,0.5) = 0.0735$ en $n=20$ y $0.07366$ en $n=80$. (c) las dos cantidades coinciden dentro del error estadístico de la segunda ($\approx 1\%$); $\mathbb E[T]\approx 0.29\,n^2 = 118$ pasos en $n=20$. (e) máximo $\approx 0.0737$ en el centro con $f = 1$ y mínimo $\approx -0.0737$ en el centro con $f=-1$; la diferencia entre las dos soluciones con y sin fuente es no negativa en todos los nodos.

In [ ]:
def valor_mc_f(i, j, M, G, F, rng=None):
    """Como valor_mc, pero acumulando h^2 F[i', j'] / 4 en cada nodo interior visitado (incluido el inicial).
    Devuelve (media, error_estandar, pasos_totales)."""
    # TODO
    pass

n = 20
G0 = np.zeros((n + 1, n + 1)); F1 = np.ones((n + 1, n + 1))
nodos = [(10, 10), (5, 5), (10, 3), (15, 10)]

# (b)
# TODO: tabla u_5p, u_MC, error estándar, z para f = 1, g = 0 (M = 20000); valor en el centro con n = 80

# (c)
# TODO: comparar 4 u_h / h^2 con pasos / M desde el centro

# (d)
# TODO: f = 40 x, g = dato_borde(20)

# (e)
# TODO: principio del máximo y de comparación con resolver (n = 40)

In [ ]:
# Verificación
assert all(abs(z) < 4 for z in z_b), f"(b) z = {z_b}"
assert all(abs(z) < 4 for z in z_d), f"(d) z = {z_d}"
assert abs(u80[40, 40] - 0.07367) < 2e-4, "el valor en el centro para n = 80 debería ser ~0.07367"
m_, e_, p_ = valor_mc_f(10, 10, 20000, G0, F1, np.random.default_rng(3))
assert abs(4 * m_ / (1 / 20) ** 2 - p_ / 20000) < 0.05 * p_ / 20000, "u = (h^2/4) E[T] con f = 1, g = 0"
assert resolver(40, np.zeros((41, 41)), 1.0)[1:-1, 1:-1].min() > 0
print("con fuente: OK")

## 6. Inpainting: rellenar una región resolviendo Laplace

### La idea

Una imagen en escala de grises es una función $I(p)$ sobre los píxeles $p$ de una grilla. Si una región $\Omega$ de la imagen se pierde (un rayón, un objeto que queremos sacar), ¿con qué la rellenamos? Una respuesta natural es: **con la función más suave que empalme con lo que rodea a la región**. "Más suave" se puede precisar como la de menor energía de gradiente,

$$\min_u \int_\Omega |\nabla u|^2\,dx\qquad\text{con } u = I \text{ en } \partial\Omega,$$

y ya sabemos (Capítulo 19) que el minimizador es la solución del problema de Dirichlet $\Delta u = 0$ en $\Omega$, $u = I$ en $\partial\Omega$: la función armónica con esos datos de borde. Discretamente, el "borde" de la región es el anillo de píxeles que la rodean, con sus valores conocidos, y cada píxel $p\in\Omega$ satisface la ecuación de promedios: $u_p$ es el promedio de sus cuatro vecinos. Es exactamente el esquema de cinco puntos, con $h = 1$ (el paso de la grilla de píxeles; no interviene, porque $f = 0$) y con un dominio de **forma arbitraria**.

### El sistema lineal sobre los píxeles de la región

Las incógnitas son sólo los $N = |\Omega|$ píxeles de la región. Numerémoslos (por ejemplo con `np.argwhere(mascara)`, que los lista por filas, y un array `idx` del tamaño de la imagen con el número de cada píxel de $\Omega$ y $-1$ afuera). Para cada píxel $p\in\Omega$ con vecinos $q\sim p$ (arriba, abajo, izquierda, derecha):

$$4\,u_p - \sum_{q\sim p,\ q\in\Omega} u_q = \sum_{q\sim p,\ q\notin\Omega} I_q .$$

Los vecinos que están dentro de la región son incógnitas (van a la matriz, con coeficiente $-1$); los que están afuera son dato (van al lado derecho). La matriz $A$ ($N\times N$) tiene $4$ en la diagonal y $-1$ en las posiciones de los vecinos internos: rala (a lo sumo $5$ por fila), simétrica y definida positiva, igual que la de la Tarea 1 pero sin la estructura de Kronecker (la región es arbitraria). Se arma con **tres listas** (filas, columnas, valores) vectorizadas por direcciones, sin bucles sobre píxeles: para cada dirección $(d_i, d_j)$ se calcula el vecino de todos los píxeles a la vez, se ve cuáles caen dentro de $\Omega$ (`idx[vecino] >= 0`), y esos aportan entradas a la matriz y los otros aportan `I[vecino]` al lado derecho. Con `sp.csr_matrix((valores, (filas, columnas)))` y `spla.spsolve` se resuelve. (La región no puede tocar el borde de la imagen: para simplificar, se pide que la máscara esté al menos a un píxel de distancia.) **Costo:** una región de $100\times100$ píxeles son $10^4$ incógnitas, milisegundos; borrar la imagen entera ($262144$ incógnitas) es un sistema ralo de tamaño $N = 512^2$, de unos segundos: la Tarea 2 con una forma distinta.

Una **prueba de escritorio** muy útil: una función *discretamente armónica* (como $I_{i,j} = i^2 - j^2$ o $I_{i,j} = ij$: comprobá que el promedio de los cuatro vecinos es igual al valor central) tiene que ser recuperada **exactamente** al borrar cualquier región y rellenarla. Si no lo es, hay un error en el armado del sistema.

### Por qué funciona en un fondo liso, y por qué no en una textura

Sea $J$ la imagen rellenada y $e = J - I$ el error dentro de $\Omega$. Como $\Delta_h J = 0$ en $\Omega$, resulta $-\Delta_h e = \Delta_h I$ en $\Omega$ con $e = 0$ en el borde: **el error es la solución de un problema de Poisson cuya fuente es el Laplaciano de la imagen original** (Tarea 6). Si el fondo es liso, $\Delta_h I\approx0$ y $e\approx0$; el error crece con la región (una región de lado $L$ tiene un error que crece como $L^2\max|\Delta_hI|$, que es de lo que viene el $0.0737\,L^2$ de $-\Delta u = 1$) y con el tamaño del Laplaciano de $I$, que en una textura (pelo, pliegues de una tela) es grande: la imagen oscila a la escala del píxel.

Y hay una razón más profunda, que es el **principio del máximo**: una función armónica no tiene máximos ni mínimos locales adentro, así que **todo lo que se rellena está entre el mínimo y el máximo del anillo**, y no puede crear ningún detalle (una mancha más clara que su entorno, una línea, un borde nuevo): sólo interpola suavemente. Lo que sale de una textura es un manchón **más suave que el entorno** (con menos desvío estándar que la imagen original), sin la variación que tenía. Y no hay parámetro que lo arregle: es una propiedad de la ecuación. (Los bordes que atraviesan la región se difuminan por lo mismo.)

**Medidas.** Para cada región conviene reportar (i) el error cuadrático medio de $J - I$ dentro de $\Omega$; (ii) el desvío estándar de $I$ y el de $J$ dentro de $\Omega$ (¿cuánto detalle se perdió?); (iii) la comparación con el relleno más tonto, la constante igual al promedio del anillo; y (iv) el chequeo del principio del máximo. Ojo: el **error cuadrático medio no es la calidad visual**: rellenar con la función suave suele tener menor error que rellenar con una textura *plausible pero distinta*, aunque la segunda "se vea mejor" (Tarea 8).

**Lectura: los métodos reales usan más.** El inpainting de la literatura reemplaza $\Delta u = 0$ por ecuaciones que propagan también las direcciones de los bordes hacia adentro de la región (Bertalmío, Sapiro, Caselles y Ballester, *Image inpainting*, SIGGRAPH 2000) o que copian texturas de otras partes de la imagen; y el *Poisson image editing* (Pérez, Gangnet y Blake, SIGGRAPH 2003) resuelve $\Delta u = \operatorname{div}\mathbf v$ con $u = I$ en el borde y un campo guía $\mathbf v$ (por ejemplo el gradiente de otra imagen), que es la Tarea 8. Todos terminan en un sistema lineal ralo como el de arriba.

In [ ]:
# La imagen y unas herramientas (esta celda viene dada)
# datos/astronauta.png: Eileen Collins (NASA), 512 x 512 en escala de grises, dominio público (datos/README.md)
I = np.asarray(plt.imread(datos.obtener("astronauta.png")), dtype=float)
if I.max() > 1.5:
    I = I / 255.0
print("imagen:", I.shape, f"valores en [{I.min():.2f}, {I.max():.2f}]")

def rect(r0, r1, c0, c1):
    """Máscara booleana del rectángulo de filas r0..r1-1 y columnas c0..c1-1."""
    m = np.zeros(I.shape, dtype=bool); m[r0:r1, c0:c1] = True
    return m

def cuadrado(centro, L):
    """Máscara del cuadrado de lado L centrado en el píxel (fila, columna) `centro`."""
    r, c = centro
    return rect(r - L // 2, r - L // 2 + L, c - L // 2, c - L // 2 + L)

def anillo(mascara):
    """Píxeles que rodean a la región (vecinos de la región que no están en ella)."""
    from scipy.ndimage import binary_dilation
    return binary_dilation(mascara) & ~mascara

### Tarea 7: Inpainting: fondo liso contra textura

**(a)** Escribí `rellenar(I, mascara)`: devuelve una copia de `I` en la que los píxeles de `mascara` se reemplazaron por la solución de $\Delta_h u = 0$ con el anillo de la región como dato (sistema ralo sobre los píxeles de la región, como en el texto de arriba; sin bucles sobre píxeles). Probala con una imagen de prueba armónica: `I_prueba[i, j] = (i**2 - j**2)/1e4 + i*j/1e4 + 0.3` de $100\times100$ con un cuadrado borrado: se tiene que recuperar exacto.

**(b)** Dos casos. **Región chica sobre fondo liso:** `mascara_liso = rect(40, 70, 315, 345)` (la pared entre la cabeza y el transbordador, arriba a la derecha; $30\times30$ píxeles). **Región grande sobre textura:** `mascara_pelo = rect(25, 95, 180, 260)` (el pelo; $70\times80$ píxeles). Para cada una calculá el relleno y armá una figura de $2\times4$ paneles (una fila por caso): la imagen original con la región marcada, la imagen con la región borrada (poné los píxeles borrados en `np.nan`, o en `1.0` si preferís, y aclará cuál), la imagen rellenada y el mapa del error $|J - I|$ dentro de la región (ojo con la escala de color del error: usá la misma para los dos casos). Mostrá un zoom de la zona (por ejemplo `I[r0-20:r1+20, c0-20:c1+20]`), no la imagen entera de $512\times 512$, así se ve el resultado.

**(c)** Para cada caso imprimí: el error cuadrático medio $\text{ECM}$ de $J - I$ en la región, el desvío estándar de $I$ y de $J$ en la región, el porcentaje de detalle perdido $1 - \operatorname{sd}(J)/\operatorname{sd}(I)$, el ECM del relleno constante (promedio del anillo) y el chequeo del principio del máximo (`J[mascara]` entre el mínimo y el máximo de `I[anillo(mascara)]`). Guardalos en el diccionario `medidas["liso"]` y `medidas["pelo"]` con las claves `"ecm"`, `"sd_I"`, `"sd_J"`, `"ecm_const"`, `"en_rango"`.

**(d)** *Tamaño de la región.* Para cuadrados de lado $L = 6, 12, 24, 48$ centrados en la pared $(80, 330)$ y en el pelo $(60, 220)$ calculá el ECM y graficalo contra $L$ (log-log, dos curvas). ¿Cómo depende del tamaño en cada caso? Guardalo en `ecm_L["pared"]` y `ecm_L["pelo"]` (listas en el orden de $L$).

**Qué se espera.** (b) En la pared el relleno es indistinguible del original a simple vista; en el pelo se ve un manchón suave, sin mechones, empalmado con el contorno. (c) Pared: $\text{ECM}\approx0.016$, $\text{sd}(I)\approx0.02$; pelo: $\text{ECM}\approx0.14$, $\text{sd}(I)\approx0.18$ y $\text{sd}(J)\approx0.09$ (se pierde la mitad del detalle); el ECM del pelo es $\approx 9$ veces el de la pared; el relleno suave le gana al constante, y en los dos casos el relleno cae dentro del rango del anillo (**es** el principio del máximo, no un accidente). (d) En la pared el error crece con $L$ pero se mantiene chico ($\approx0.005$ con $L=6$ y $\approx0.02$ con $L=48$); en el pelo ya es grande con $L = 6$ ($\approx0.06$: una región de $6\times6$ dentro de una textura no se puede adivinar) y llega a $\approx0.15$ con $L=48$, que es más o menos el desvío de la textura: a partir de cierto tamaño rellenar con Laplace no mejora al relleno constante.

In [ ]:
def rellenar(I, mascara):
    """Copia de I con los píxeles de `mascara` rellenados resolviendo Laplace discreto con el anillo como dato."""
    # TODO: numerar los píxeles de la región (idx), armar filas/columnas/valores por dirección,
    #       lado derecho con los vecinos que están fuera de la región, csr_matrix + spsolve
    pass

# (a) prueba con una imagen discretamente armónica
ii, jj = np.meshgrid(np.arange(100), np.arange(100), indexing="ij")
I_prueba = (ii ** 2 - jj ** 2) / 1e4 + ii * jj / 1e4 + 0.3
# TODO: borrar un cuadrado (por ejemplo filas 30..59, columnas 40..69) y verificar que se recupera

# (b)
mascara_liso = rect(40, 70, 315, 345)
mascara_pelo = rect(25, 95, 180, 260)
# TODO: rellenar los dos casos y la figura de 2 x 4 paneles (con zoom)

# (c)
medidas = {"liso": {}, "pelo": {}}    # TODO: claves "ecm", "sd_I", "sd_J", "ecm_const", "en_rango"

# (d)
Ls = [6, 12, 24, 48]
ecm_L = {"pared": [], "pelo": []}     # TODO: ECM para cada L (cuadrados centrados en (80, 330) y (60, 220))
# TODO: figura log-log

In [ ]:
# Verificación
mp = np.zeros((100, 100), dtype=bool); mp[30:60, 40:70] = True
assert np.abs(rellenar(I_prueba, mp) - I_prueba).max() < 1e-9, "la imagen armónica de prueba no se recupera exacta"
assert np.abs(rellenar(I, mascara_liso)[~mascara_liso] - I[~mascara_liso]).max() == 0, "fuera de la región no se toca nada"
m1, m2 = medidas["liso"], medidas["pelo"]
assert m1["ecm"] < 0.03 and m2["ecm"] > 3 * m1["ecm"], f"ECM liso {m1['ecm']:.3f}, pelo {m2['ecm']:.3f}: el del pelo debería ser mucho mayor"
assert m2["sd_J"] < 0.75 * m2["sd_I"], "en la textura el relleno debería ser más suave que el original"
assert m1["en_rango"] and m2["en_rango"], "el relleno debe caer dentro del rango del anillo (principio del máximo)"
assert all(a < b for a, b in zip(ecm_L["pared"], ecm_L["pelo"])), "para cada L el error en el pelo debería ser mayor"
print("inpainting: OK")

### (Opcional) Rellenar con una fuente: Poisson guiado y clonado de textura

La solución de Laplace es la más suave posible: no puede crear detalle. Para conseguir **detalle** hay que dárselo, y la ecuación de Poisson es el modo natural: si en lugar de $\Delta u = 0$ resolvemos

$$\Delta u = \Delta S\ \text{ en }\Omega,\qquad u = I\ \text{ en }\partial\Omega,$$

donde $S$ es **otra imagen** (o la misma imagen desplazada), $u$ tiene, dentro de $\Omega$, el mismo Laplaciano —el mismo "detalle"— que $S$, y empalma con $I$ en el borde: es la solución de $\min\int_\Omega|\nabla u - \nabla S|^2$ con $u=I$ en $\partial\Omega$, es decir, la imagen que **más se parece en gradientes a $S$ y coincide con $I$ en el borde**. La costura desaparece porque el dato de borde es el de $I$: las diferencias de brillo entre $S$ e $I$ se reparten suavemente por la región. Esto es el *Poisson image editing*. En el lenguaje de la Sección 5, es $-\Delta u = f$ con $f = -\Delta_h S$. Discretamente, la ecuación del píxel $p\in\Omega$ es

$$4u_p - \sum_{q\sim p,\ q\in\Omega}u_q = \sum_{q\sim p,\ q\notin\Omega}I_q - (\Delta_h S)_p,\qquad (\Delta_h S)_p = \sum_{q\sim p}S_q - 4S_p ,$$

el mismo sistema que en la Tarea 7 con un término más en el lado derecho. **Dos pruebas de escritorio:** si $S = I$ (la guía es la propia imagen), $\Delta_hu = \Delta_hI$ con el mismo borde: $u = I$, se recupera la imagen exacta; y si $S$ es constante, $\Delta_hS=0$ y volvemos a Laplace. Como $f\ne0$, **el principio del máximo ya no vale**: el relleno puede salirse del rango del anillo (y es lo que hace falta para agregar detalle).

### Tarea 8: (Opcional) Poisson guiado: clonar textura

**(a)** Escribí `rellenar_guiado(I, mascara, guia)`: como `rellenar`, pero con la guía `guia` (un array del tamaño de la imagen) y el lado derecho corregido con $-(\Delta_h S)_p$. Verificá las dos pruebas de escritorio: con `guia = I` se recupera `I` exacta, y con `guia = np.full(I.shape, 0.5)` da lo mismo que `rellenar`.

**(b)** Rellená el pelo (`mascara_pelo` de la Tarea 7) con la guía `S = np.roll(I, (0, -30), axis=(0, 1))` (o sea `S[r, c] = I[r, c + 30]`: el pelo 30 píxeles a la derecha, es decir, la propia textura del pelo desplazada). Mostrá en una figura (con zoom) el original, el relleno de Laplace (Tarea 7), el relleno guiado y el mapa de $|J-I|$ del guiado, y calculá: el ECM del guiado, el desvío estándar del relleno guiado dentro de la región y el chequeo del principio del máximo. Guardalos en `medidas_g` con las claves `"ecm"`, `"sd_J"` y `"en_rango"`.

**(c)** Probá otra guía a tu elección (otro desplazamiento, la imagen espejada, otra parte del cuerpo...) y anotá qué pasa cuando la textura de la guía no se parece a la del entorno.

**Qué se espera.** (a) errores $\lesssim10^{-9}$. (b) el relleno guiado **se ve mucho mejor**: tiene mechones y contraste, y empalma con el contorno sin costura; pero su ECM contra el original es **mayor** que el de Laplace ($\approx0.21$ contra $\approx0.14$): la textura plausible pero *desalineada* con la real ($30$ píxeles corridos) tiene un error de amplitud comparable a la de la propia textura, y el suave, que es el "promedio" de todas las posibilidades, minimiza el error cuadrático. El desvío estándar del relleno guiado ($\approx 0.15$) es mucho más parecido al del original ($\approx0.17$) que el de Laplace ($\approx0.09$), y el chequeo del principio del máximo **falla** (el relleno se sale del rango del anillo, porque $f\neq0$). Ojo con esa lección: el ECM mide *cuánto se parece al original*, no *cuán creíble es el resultado*.

In [ ]:
def rellenar_guiado(I, mascara, guia):
    """Como rellenar, pero con Delta u = Delta guia en la región (Poisson guiado)."""
    # TODO: igual que rellenar, restando (Delta_h guia)[p] = sum(guia de los 4 vecinos) - 4 guia[p] del lado derecho
    pass

# (a)
# TODO: pruebas de escritorio (guia = I; guia constante)

# (b)
guia = np.roll(I, (0, -30), axis=(0, 1))
medidas_g = {}       # TODO: claves "ecm", "sd_J", "en_rango"
# TODO: figura (zoom): original, Laplace, guiado, error del guiado

# (c)
# TODO: otra guía a elección

In [ ]:
# Verificación
assert np.abs(rellenar_guiado(I, mascara_pelo, I) - I).max() < 1e-9, "con guia = I se debería recuperar la imagen"
assert np.abs(rellenar_guiado(I, mascara_pelo, np.full(I.shape, 0.5)) - rellenar(I, mascara_pelo)).max() < 1e-9, "con guia constante debería dar Laplace"
assert medidas_g["sd_J"] > 1.3 * medidas["pelo"]["sd_J"] and not medidas_g["en_rango"], "el guiado tiene detalle y se sale del rango del anillo"
print("Poisson guiado: OK")

## Interpretación

Esta es la parte que se evalúa. Respondé en las celdas de texto que siguen, con oraciones completas y citando los números o gráficos que obtuviste.

**1.** **Tres maneras de resolver el mismo sistema.** Con los números de las Tareas 2 y 5 (exponentes de tiempo y memoria, iteraciones contra $n$), comparé la matriz densa, la factorización rala y los métodos iterativos (Jacobi, Gauss–Seidel, gradiente conjugado): ¿cómo escalan el tiempo y la memoria de cada uno con $n$? ¿Por qué Jacobi necesita $\sim n^2$ iteraciones (usá la lectura probabilística: ¿cuánto tarda un paseo en llegar al borde?) y qué gana el gradiente conjugado? ¿Qué método elegirías para $n = 100$, y cuál para un problema de $10^7$ incógnitas en tres dimensiones?

*(escribí tu respuesta acá)*

**2.** **Cinco puntos contra Monte Carlo.** Con las Tareas 3, 4 y 6: ¿cómo decrece el error de Monte Carlo con $M$ (pendiente medida) y de qué depende la constante? Distinguí el **error de discretización** (el del esquema, $O(h^2)$, igual en los dos métodos) del **error estadístico**, y explicá por qué un aumento de $M$ no mejora el primero. Para el mismo error de $10^{-2}$ en un nodo, ¿cuál método es más caro y por cuánto (tabla de la Tarea 4)? ¿Y para todos los nodos? ¿Cuándo le ganaría Monte Carlo (mirá el experimento en dimensión $d$: qué cambia y qué no con $d$)? Indicá qué le falta a Monte Carlo en 2D que sí tiene el sistema lineal.

*(escribí tu respuesta acá)*

**3.** **La fuente y el principio del máximo.** Explicá cómo se modifica el juego para resolver $-\Delta u = f$ (qué se cobra, cuándo, y por qué el factor $h^2/4$), y qué verificaste numéricamente (Tarea 6: $z$-valores, relación con el tiempo de salida). ¿Qué dice el principio del máximo sobre $u$ cuando $f = 0$, y qué pasa cuando $f\ge0$ (mínimo, máximo, comparación)? Contá cómo lo usaste como **diagnóstico** de un solver (Tarea 1(d)): ¿qué errores detecta y cuáles no?

*(escribí tu respuesta acá)*

**4.** **Inpainting.** Con las Tareas 7 y 8: ¿por qué rellenar con la función armónica funciona en un fondo liso y no en una textura? Usá el principio del máximo (¿qué puede y qué no puede aparecer dentro de la región?) y los números de la Tarea 7 (ECM y desvío estándar del original y del relleno en la pared y en el pelo; el efecto del tamaño de la región). Explicá por qué el error del relleno es la solución de un problema de Poisson con fuente $\Delta_hI$. Si hiciste la Tarea 8: ¿por qué el Poisson guiado se ve mejor y tiene *peor* error cuadrático, y qué te dice eso sobre usar el ECM como medida de calidad de una imagen?

*(escribí tu respuesta acá)*